Readability average scores calculation script:
----------------------------------------------
This script calculates the averages for the readability analysis scores. It's applicable for ground truth human annotations, LLM analysis on ground truth sample, and final LLM analysis dataset.

In [1]:
import pandas as pd
import os

Configuration:

In [ ]:
# File names and directories
ROOT = "--FILL IN YOUR ROOT FOLDER--"  # root project folder
RDBL_DIR = f"{ROOT}/LLM Analyses/Readability Analysis/Ground Truth sample - rdbl/Readability analyses"
GROUND_TRUTH_DIR = f"{ROOT}/Ground Truth/Annotations"
OUTPUT_DIR = {"ground truth LLM": RDBL_DIR, # choose output type
               "ground truth": GROUND_TRUTH_DIR,
                 "final analysis": f"{ROOT}/LLM Analyses/Readability Analysis"}["ground truth"]

GROUND_TRUTH_FILE = "Ground truth - rdbl.csv"  # Manually classified csv

# The 12 adjusted Shimamura et al. readability criteria
SUBCRITERIA = [
    "A-01", "A-02", "B-01", "B-02", "B-03", 
    "C-01", "C-02", "C-04", "C-05", "C-06", "D-01",
    "L-01"
]

# Mapping to main criteria for calculating the averages
PARENT_MAP = {
    "A-01": "A", "A-02": "A",
    "B-01": "B", "B-02": "B", "B-03": "B",
    "C-01": "C", "C-02": "C", "C-04": "C", "C-05": "C", "C-06": "C",
    "D-01": "D",
    "L-01": "L"
}

Main algorithm:

In [ ]:
def calculate(input_file, gt=False):
    df = pd.read_csv(input_file)
    # A. Initialize Columns
    # Initialize Word_Count column, if this one already exists in the dataset, then the file is skipped
    if "word_count" not in df.columns: 
        df["word_count"] = None
    else:
        print(f"{input_file.split("\\")[-1]}: scores already calculated")
        return

    # Initialize Averages
    for main_crit in ["Crit_A", "Crit_B", "Crit_C", "Crit_D", "Crit_L"]:
        if main_crit not in df.columns: df[main_crit] = None
    if "AverageRDBL" not in df.columns: df["AverageRDBL"] = None

    # Calculate range
    end_index = len(df)

    print(f"Processing posts from index 0 to {end_index}...")

    # B. Processing Loop
    for index in range(end_index):
        post_content = str(df.at[index, "Text"])
        
         # Skip if not successfully processed yet (unless it's the ground truth file)
        if gt or (pd.notna(df.at[index, 'Gunning_Fog']) and "ERROR" not in str(df.at[index, 'Gunning_Fog'])):

            # 1. Calculate word count
            df.at[index, "word_count"] = len(post_content.split())

            # 3. Map Scores and Calculate Averages per Main Criteria
            crit_scores = {"A": [], "B": [], "C": [], "D": [], "L": []} # for the average score per criteria
            total_scores = 0
            n_scores = 0

            for sub in SUBCRITERIA:
                clean_sub = sub.replace("-", "_")
                score_val = df.at[index, f"{clean_sub}_score"]

                # Prep for Averages (if numeric)
                if score_val and pd.notna(score_val):
                    try:
                        num_score = float(score_val)
                        parent = PARENT_MAP[sub]
                        crit_scores[parent].append(num_score)
                        total_scores += num_score
                        n_scores += 1
                    except ValueError:
                        pass 
            # 4. Compute and save Averages
            for cat_prefix, scores in crit_scores.items():
                if scores:
                    av = sum(scores) / len(scores)
                    df.at[index, f"Crit_{cat_prefix}"] = av

            df.at[index, "Average_Subcrit"] = total_scores / n_scores if n_scores else 0

    # Final Save
    df.to_csv(input_file, index=False, encoding='utf-8-sig')
    print(f"\nProcessing complete! Results saved to {input_file}")

def main():
    """Iterates over all readability analysis outputs in given directory"""
    if OUTPUT_DIR == GROUND_TRUTH_DIR:
        calculate(os.path.join(GROUND_TRUTH_DIR, GROUND_TRUTH_FILE), True)
    else:   # For final analysis and ground truth analyses
        for rdbl_file in os.listdir(OUTPUT_DIR):
            if "Readability analysis_" in rdbl_file:
                calculate(os.path.join(OUTPUT_DIR, rdbl_file))
    print("Scores calculated.")

if __name__ == "__main__":
    main()